In [2]:
import torch
import torchvision.transforms as transforms
from torchvision.models import resnet50
from PIL import Image
import numpy as np
from tqdm import tqdm


In [ ]:
#model now outputs a 2048-dimensional vector that represents the "essence" of the image

device = "cuda" if torch.cuda.is_available() else "cpu"

model = resnet50(pretrained=True)
model.fc = torch.nn.Identity()   # remove classification head
model = model.to(device)
model.eval()


d:\MINI PROJECT\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\MINI PROJECT\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to C:\Users\Admin/.cache\torch\hub\checkpoints\resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [01:36<00:00, 1.06MB/s]


ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [5]:
def extract_poster_embedding(image_path):
    img = Image.open(image_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        embedding = model(img)

    return embedding.cpu().numpy().flatten()


In [7]:
import pandas as pd
movies=pd.read_csv("D:\MINI PROJECT\DATASET\movies_final.csv")

In [8]:
poster_embeddings = {}
for movie_id in tqdm(movies["movieId"]):
    poster_path = f"posters/{movie_id}.jpg"

    try:
        emb = extract_poster_embedding(poster_path)
        poster_embeddings[movie_id] = emb
    except:
        continue  # missing poster


  0%|          | 0/23138 [00:00<?, ?it/s]

100%|██████████| 23138/23138 [00:00<00:00, 65305.22it/s]


In [9]:
poster_embedding_matrix = []

for mid in movies["movieId"]:
    if mid in poster_embeddings:
        poster_embedding_matrix.append(poster_embeddings[mid])
    else:
        poster_embedding_matrix.append(np.zeros(2048))  # fallback


In [10]:
poster_embedding_matrix = np.array(poster_embedding_matrix)
np.save("poster_embeddings.npy", poster_embedding_matrix)


CHECKING OUTPUT EMBEDDINGS

In [11]:
import torch
from torchvision.models import resnet50
from torchvision import transforms
from PIL import Image
import numpy as np


In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = resnet50(pretrained=True)
model.fc = torch.nn.Identity()   # remove classification head
model = model.to(device)
model.eval()

print("Model loaded on:", device)


d:\MINI PROJECT\venv\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\MINI PROJECT\venv\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model loaded on: cpu


In [13]:
image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [16]:
img_path = r"D:\MINI PROJECT\images.jpeg"
img = Image.open(img_path).convert("RGB")
img_tensor = image_transform(img).unsqueeze(0).to(device)

print("Image tensor shape:", img_tensor.shape)


Image tensor shape: torch.Size([1, 3, 224, 224])


In [17]:
with torch.no_grad():
    embedding = model(img_tensor)

print("Poster embedding shape:", embedding.shape)


Poster embedding shape: torch.Size([1, 2048])


In [18]:
poster_embedding_np = embedding.cpu().numpy().flatten()
print(poster_embedding_np.shape)


(2048,)
